# Combinatorial Generation

Enumerating all **subsets**, **permutations**, or **combinations** of a set — the search-space patterns. This notebook **consolidates** machinery from earlier tiers: subsets via **backtracking** (recursion notebook) and via **bitmasks** (bit-manipulation notebook), framed as one reusable pattern — and shows the `itertools` that does it all for you in practice.

> signal → template → worked examples → toolkit

**Contents**

1. **The signal**
2. **Subsets** — backtracking & bitmask
3. **Permutations & combinations**
4. **The Python toolkit** — `itertools`
5. **When to use & cheat-sheet**

## 1. The signal — enumerate a combinatorial space

Reach for these when a problem asks you to **generate or count every** arrangement: all subsets (the power set), all orderings (permutations), all size-k selections (combinations), or the Cartesian product of several choices.

Two engines build them:

- **Backtracking** (the recursion notebook's *choose → explore → un-choose*) — general, works for all four, and lets you **prune** invalid branches early.
- **Bitmasks** (the bit-manipulation notebook) — for **subsets** specifically, each of the `2^n` integers `0 .. 2^n-1` *is* a subset (bit i = "item i included").

And the Python reality: **`itertools` already implements all of them in C**, so you reach for a hand-rolled version mainly to *prune*, or to learn the shape.

## 2. Subsets — backtracking & bitmask

A set of n items has **2ⁿ** subsets. Two ways to emit them all.

**Backtracking** grows a path, records it at every node, and only ever extends *forward* (so no duplicates):

In [1]:
def subsets_backtrack(items):
    res, path = [], []
    def backtrack(start):
        res.append(path[:])                  # every node of the tree is a subset
        for i in range(start, len(items)):
            path.append(items[i])            # choose
            backtrack(i + 1)                 # explore (forward only)
            path.pop()                       # un-choose
    backtrack(0)
    return res

print("subsets_backtrack([1,2,3]):", subsets_backtrack([1, 2, 3]))


subsets_backtrack([1,2,3]): [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]


**Bitmask** treats each integer `0 .. 2ⁿ-1` as a membership pattern — bit i set means item i is in the subset:

In [2]:
def subsets_bitmask(items):
    n = len(items)
    return [[items[i] for i in range(n) if (mask >> i) & 1]
            for mask in range(1 << n)]

print("subsets_bitmask([1,2,3]):", subsets_bitmask([1, 2, 3]))


subsets_bitmask([1,2,3]): [[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]


Same 2ⁿ subsets, different order. The bitmask form is the basis of **bitmask DP** (a state *is* a subset of items); backtracking is the one to use when you must **prune** subtrees — e.g. "subsets summing to ≤ target," where you abandon a branch the moment it overflows.

## 3. Permutations & combinations

**Permutations** (all n! orderings) use a `used[]` flag so each item appears once per arrangement; **combinations** (all C(n, k) size-k selections) reuse the forward-only `start` index from subsets but only record at depth k:

In [3]:
def permutations(items):
    res, used, path = [], [False] * len(items), []
    def backtrack():
        if len(path) == len(items):
            res.append(path[:])
            return
        for i in range(len(items)):
            if used[i]:
                continue
            used[i] = True; path.append(items[i])    # choose
            backtrack()                              # explore
            path.pop(); used[i] = False              # un-choose
    backtrack()
    return res

def combinations(items, k):
    res, path = [], []
    def backtrack(start):
        if len(path) == k:
            res.append(path[:])
            return
        for i in range(start, len(items)):
            path.append(items[i])
            backtrack(i + 1)                         # forward only -> no repeats
            path.pop()
    backtrack(0)
    return res

print("permutations([1,2,3])     :", permutations([1, 2, 3]))
print("combinations([1,2,3,4], 2):", combinations([1, 2, 3, 4], 2))


permutations([1,2,3])     : [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
combinations([1,2,3,4], 2): [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]


These are exactly the templates from the recursion-and-backtracking notebook (§5). The only difference between subsets / combinations / permutations is *when you record* a result and *whether you reuse* elements — one skeleton, three behaviors.

## 4. The Python toolkit — `itertools`

In real code you almost never hand-roll these — `itertools` generates them **lazily in C**: faster, and O(1) memory until you actually consume them:

In [4]:
from itertools import combinations, permutations, product, combinations_with_replacement, chain

print("combinations([1,2,3,4], 2)      :", list(combinations([1, 2, 3, 4], 2)))
print("permutations([1,2,3])           :", list(permutations([1, 2, 3])))
print("product([0,1], repeat=2)        :", list(product([0, 1], repeat=2)))   # all bit patterns
print("combinations_w_replacement(.,2) :", list(combinations_with_replacement([1, 2, 3], 2)))

# the power set is a one-liner with chain + combinations:
def powerset(items):
    return list(chain.from_iterable(combinations(items, r) for r in range(len(items) + 1)))

print("powerset([1,2,3])               :", powerset([1, 2, 3]))


combinations([1,2,3,4], 2)      : [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
permutations([1,2,3])           : [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
product([0,1], repeat=2)        : [(0, 0), (0, 1), (1, 0), (1, 1)]
combinations_w_replacement(.,2) : [(1, 1), (1, 2), (1, 3), (2, 2), (2, 3), (3, 3)]
powerset([1,2,3])               : [(), (1,), (2,), (3,), (1, 2), (1, 3), (2, 3), (1, 2, 3)]


`product(seq, repeat=n)` is the Cartesian product — `product([0,1], repeat=n)` enumerates all n-bit patterns (the bitmask subsets, as tuples). Use the hand-written backtracking only when you need to **prune** mid-generation, which `itertools` can't do for you.

## 5. When to use & cheat-sheet

| You need… | Use |
|---|---|
| all subsets (power set) | bitmask `range(1<<n)`, or `chain` + `combinations` |
| all subsets, with pruning | backtracking |
| all orderings | `itertools.permutations` (or backtracking to prune) |
| all size-k selections | `itertools.combinations` |
| every mix of independent choices | `itertools.product` |
| just the **count** | a formula — `2**n`, `math.factorial(n)`, `math.comb(n,k)` — don't generate! |

**The size warning:** these grow **exponentially** — 2ⁿ subsets, n! permutations. Generating is fine for small n (≲ 20 for subsets, ≲ 10 for permutations); beyond that you must **prune** (backtracking + a feasibility check), switch to **bitmask DP**, or just **count** with a formula instead of enumerating.

| Structure | Count | Generator |
|---|:---:|---|
| Subsets | 2ⁿ | bitmask / backtracking / `chain`+`combinations` |
| Permutations | n! | `itertools.permutations` / backtracking |
| Combinations (choose k) | `math.comb(n, k)` | `itertools.combinations` |
| Product (m choices, n slots) | mⁿ | `itertools.product(…, repeat=n)` |